# Cross-Domain Evaluation (Generalizability)
This notebook addresses the reviewer's comment regarding generalizability by evaluating best checkpoints (Transformer, ML, Sequence) trained on **Twitter** and **YouTube** against three unseen domains: **Amazon**, **Governmental**, and **News**.


In [ ]:
!pip install transformers arabert gensim emoji openpyxl -q
from transformers import AutoModelForSequenceClassification, AutoTokenizer


In [ ]:
# 1. Setup & Drive Mounting
import json
import os
import re
import string
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import emoji
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from gensim.models import FastText, Word2Vec
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset as TorchDataset

# The evaluation notebook runs from the project root locally or in Colab.

warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / "Dataset").exists():
    raise FileNotFoundError(f"Run from the project root; Dataset/ was not found under {BASE_DIR}")
OUTPUT_DIR = BASE_DIR / "Output"
DATA_DIR = BASE_DIR / "Dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")
print(f"BASE_DIR: {BASE_DIR}")


In [ ]:
# 2. Load New Datasets + Source Training Splits (for exact preprocessing parity)
DATASET_PATHS = {
    "Amazon": DATA_DIR / "Amazon_data" / "Amazon_data" / "splits" / "test.csv",
    "Governmental": DATA_DIR / "governmental_data" / "governmental_data" / "splits" / "test.csv",
    "News": DATA_DIR / "News_data" / "News_data" / "splits_rebuilt" / "test.csv",
}

SOURCE_TRAIN_PATHS = {
    "twitter": DATA_DIR / "Twitter_data" / "Twitter_data" / "splits_rebuilt_v4" / "train_with_meta.csv",
    "youtube": DATA_DIR / "Youtube_data" / "Youtube_data" / "splits_rebuilt" / "train_with_meta.csv",
}

def _normalize_label(value):
    if pd.isna(value):
        return 0
    if isinstance(value, (bool, np.bool_)):
        return int(value)
    if isinstance(value, (int, np.integer, float, np.floating)):
        return int(round(float(value)))
    token = str(value).strip().lower()
    if token in {"1", "true", "fake", "deceptive", "yes", "y"}:
        return 1
    if token in {"0", "false", "real", "genuine", "no", "n"}:
        return 0
    try:
        return int(float(token))
    except Exception:
        return 0

def _load_eval_datasets():
    datasets = {}
    for name, path in DATASET_PATHS.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing dataset file: {path}")
        df = pd.read_csv(path)
        if "text" not in df.columns or "label" not in df.columns:
            raise ValueError(f"{name} must contain 'text' and 'label' columns: {path}")
        out = df[["text", "label"]].copy()
        out["text"] = out["text"].fillna("").astype(str)
        out["label"] = out["label"].apply(_normalize_label).astype(int)
        datasets[name] = out
    return datasets

def _load_source_train_texts(path):
    if not path.exists():
        print(f"Warning: source train split not found -> {path}")
        return pd.Series([], dtype=str)
    df = pd.read_csv(path)
    if "text" not in df.columns:
        return pd.Series([], dtype=str)
    return df["text"].fillna("").astype(str)

datasets = _load_eval_datasets()
source_train_texts = {platform: _load_source_train_texts(path) for platform, path in SOURCE_TRAIN_PATHS.items()}

for name, df in datasets.items():
    print(f"{name}: {len(df)} rows | labels={sorted(df['label'].unique().tolist())}")
for platform, texts in source_train_texts.items():
    print(f"{platform} source train rows for odd-char extraction: {len(texts)}")


In [ ]:
# 3. Preprocessing Functions (aligned with training notebooks)

SPECIAL_TOKENS = {
    "url": "<URL>",
    "mention": "<MENTION>",
    "hashtag": "<HASHTAG>",
    "number": "<NUMBER>",
    "latin": "<LATIN>",

}

SPECIAL_TOKEN_PATTERN = re.compile(r"(<URL>|<MENTION>|<HASHTAG>|<NUMBER>|<LATIN>)")

ARABIC_DIACRITICS_PATTERN = re.compile(r"[\u064B-\u065F\u0670]")

TATWEEL_CHAR = "\u0640"

ARABIC_INDIC_DIGITS = "٠١٢٣٤٥٦٧٨٩"

EASTERN_ARABIC_DIGITS = "۰۱۲۳۴۵۶۷۸۹"

ALEF_VARIANTS = {"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا"}

YEH_VARIANTS = {"ى": "ي", "ی": "ي"}

HAMZA_VARIANTS = {"ؤ": "و", "ئ": "ي"}

URL_PATTERN = re.compile(r"(?:https?://\S+|www\.\S+)", flags=re.IGNORECASE)

MENTION_PATTERN = re.compile(r"(?<!\w)@[A-Za-z0-9_]+")

HASHTAG_PATTERN = re.compile(r"(?<!\w)#[\w\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF_]+")

ENGLISH_PATTERN = re.compile(r"[A-Za-z]+(?:[A-Za-z0-9_./+-]*[A-Za-z0-9])?")

NUMBER_SPAN_PATTERN = re.compile(r"(?<!\w)[\d\u0660-\u0669\u06F0-\u06F9]+(?:[.,:/\-][\d\u0660-\u0669\u06F0-\u06F9]+)*(?:[%٪])?")

HTML_TAG_PATTERN = re.compile(r"</?[^>\n]+>")

HTML_ENTITY_PATTERN = re.compile(r"&(?:[A-Za-z]+|#\d+|#x[0-9A-Fa-f]+);")

INVISIBLE_FORMATTING_PATTERN = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2060-\u206f\ufeff]")

REPEATED_SEPARATOR_PATTERN = re.compile(r"([\-_=+~*#|/\\])\1+")

MINIMAL_PUNCT = "،.؟"

ASCII_PUNCT = string.punctuation

def is_arabic_char(ch):
    code = ord(ch)
    return (
        (0x0600 <= code <= 0x06FF) or
        (0x0750 <= code <= 0x077F) or
        (0x08A0 <= code <= 0x08FF) or
        (0xFB50 <= code <= 0xFDFF) or
        (0xFE70 <= code <= 0xFEFF)
    )

def is_english_char(ch):
    return ("A" <= ch <= "Z") or ("a" <= ch <= "z")

def safe_str(text):
    if pd.isna(text):
        return ""
    return str(text)


In [ ]:
def strip_html_like(text):
    text = safe_str(text)
    text = HTML_TAG_PATTERN.sub(" ", text)
    return HTML_ENTITY_PATTERN.sub(" ", text)

def remove_invisible_and_control_chars(text):
    cleaned = []
    for ch in safe_str(text):
        if INVISIBLE_FORMATTING_PATTERN.match(ch):
            continue
        if unicodedata.category(ch) in {"Cf", "Cc", "Cs"} and ch not in {"\n", "\t", "\r"}:
            continue
        cleaned.append(ch)
    return "".join(cleaned)

def normalize_alef(text):
    text = safe_str(text)
    for variant, normalized in ALEF_VARIANTS.items():
        text = text.replace(variant, normalized)
    return text

def normalize_yeh(text):
    text = safe_str(text)
    for variant, normalized in YEH_VARIANTS.items():
        text = text.replace(variant, normalized)
    return text

def normalize_hamza(text):
    text = safe_str(text)
    for variant, normalized in HAMZA_VARIANTS.items():
        text = text.replace(variant, normalized)
    return text

def remove_diacritics(text):
    return ARABIC_DIACRITICS_PATTERN.sub("", safe_str(text))

def remove_tatweel(text):
    return safe_str(text).replace(TATWEEL_CHAR, "")

def normalize_repetition(text, max_repeat=2):
    text = safe_str(text)
    pattern = r"(.)\1{" + str(max_repeat) + r",}"
    return re.sub(pattern, r"\1" * max_repeat, text)

def apply_outside_special_tokens(text, transform):
    text = safe_str(text)
    parts = SPECIAL_TOKEN_PATTERN.split(text)
    out = []
    for part in parts:
        if not part:
            continue
        if SPECIAL_TOKEN_PATTERN.fullmatch(part):
            out.append(part)
        else:
            out.append(transform(part))
    return "".join(out)

def normalize_repetition_keep_tokens(text, max_repeat=2):
    return apply_outside_special_tokens(text, lambda part: normalize_repetition(part, max_repeat=max_repeat))

PUNCT_TRANSLATION = str.maketrans({
    ",": "،",
    ";": "،",
    "؛": "،",
    "?": "؟",
    "!": ".",
    "…": ".",

})


In [ ]:
def normalize_punctuation(text):
    text = safe_str(text).translate(PUNCT_TRANSLATION)
    text = re.sub(r"،{2,}", "،", text)
    text = re.sub(r"\.{2,}", ".", text)
    text = re.sub(r"؟{2,}", "؟", text)
    return text

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", safe_str(text)).strip()

def normalize_token_spacing(text):
    text = SPECIAL_TOKEN_PATTERN.sub(lambda m: f" {m.group(1)} ", safe_str(text))
    text = re.sub(r"\s+([،.؟])", r"\1", text)
    text = re.sub(r"([،.؟])(?=\S)", r"\1 ", text)
    return normalize_whitespace(text)

def replace_urls(text):
    return URL_PATTERN.sub(f" {SPECIAL_TOKENS['url']} ", safe_str(text))

def replace_mentions(text):
    return MENTION_PATTERN.sub(f" {SPECIAL_TOKENS['mention']} ", safe_str(text))

def replace_hashtags(text):
    return HASHTAG_PATTERN.sub(f" {SPECIAL_TOKENS['hashtag']} ", safe_str(text))

def remove_urls(text):
    return URL_PATTERN.sub(" ", safe_str(text))

def remove_mentions(text):
    return MENTION_PATTERN.sub(" ", safe_str(text))

def remove_hashtags(text):
    return HASHTAG_PATTERN.sub(" ", safe_str(text))

def replace_number_spans(text):
    return apply_outside_special_tokens(text, lambda part: NUMBER_SPAN_PATTERN.sub(f" {SPECIAL_TOKENS['number']} ", part))

def replace_latin_spans(text):
    return apply_outside_special_tokens(text, lambda part: ENGLISH_PATTERN.sub(f" {SPECIAL_TOKENS['latin']} ", part))

def remove_latin_spans(text):
    return apply_outside_special_tokens(text, lambda part: ENGLISH_PATTERN.sub(" ", part))

def remove_emojis(text):
    return emoji.replace_emoji(safe_str(text), replace=" ")

def collapse_visual_separators(text):
    return apply_outside_special_tokens(text, lambda part: REPEATED_SEPARATOR_PATTERN.sub(" ", part))

def filter_text_chars(text, keep_tokens=False):
    text = safe_str(text)
    parts = SPECIAL_TOKEN_PATTERN.split(text) if keep_tokens else [text]
    out = []
    for part in parts:
        if keep_tokens and SPECIAL_TOKEN_PATTERN.fullmatch(part):
            out.append(part)
            continue
        kept = []
        for ch in part:
            if ch.isspace():
                kept.append(ch)
            elif is_arabic_char(ch) or ch.isdigit() or ch in MINIMAL_PUNCT:
                kept.append(ch)
        out.append("".join(kept))
    return "".join(out)


In [ ]:
def extract_odd_chars(texts, keep_chars=None):
    keep = set(keep_chars or [])
    odd = set()
    for text in texts.fillna(""):
        for ch in str(text):
            if ch.isspace() or ch in keep:
                continue
            if is_arabic_char(ch) or is_english_char(ch) or ch.isdigit():
                continue
            odd.add(ch)
    return "".join(sorted(odd))

def compile_odd_pattern(odd_chars):
    if not odd_chars:
        return None
    return re.compile("[" + re.escape(odd_chars) + "]")

def handle_odd_chars(text, pattern=None):
    text = safe_str(text)
    if pattern is None or not pattern.search(text):
        return text
    return normalize_whitespace(pattern.sub(" ", text))

ODD_CHARS_PATTERNS = {
    "twitter": compile_odd_pattern(extract_odd_chars(source_train_texts["twitter"], keep_chars=ASCII_PUNCT)),
    "youtube": compile_odd_pattern(extract_odd_chars(source_train_texts["youtube"], keep_chars=ASCII_PUNCT)),

}

def preprocess_manual_twitter(text):
    text = safe_str(text)
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = strip_html_like(text)
    text = remove_invisible_and_control_chars(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = remove_hashtags(text)
    text = remove_emojis(text)
    text = remove_diacritics(text)
    text = normalize_alef(text)
    text = normalize_yeh(text)
    text = normalize_hamza(text)
    text = remove_tatweel(text)
    text = normalize_repetition(text, max_repeat=1)
    text = normalize_punctuation(text)
    text = collapse_visual_separators(text)
    text = handle_odd_chars(text, pattern=ODD_CHARS_PATTERNS["twitter"])
    text = remove_latin_spans(text)
    text = filter_text_chars(text, keep_tokens=False)
    text = normalize_token_spacing(text)
    return text


In [ ]:
def preprocess_deepfake_aware_twitter(text):
    text = safe_str(text)
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = strip_html_like(text)
    text = remove_invisible_and_control_chars(text)
    text = replace_urls(text)
    text = replace_mentions(text)
    text = replace_hashtags(text)
    text = replace_number_spans(text)
    text = replace_latin_spans(text)
    text = remove_emojis(text)
    text = normalize_alef(text)
    text = normalize_yeh(text)
    text = normalize_hamza(text)
    text = remove_tatweel(text)
    text = normalize_repetition_keep_tokens(text, max_repeat=2)
    text = normalize_punctuation(text)
    text = collapse_visual_separators(text)
    text = handle_odd_chars(text, pattern=ODD_CHARS_PATTERNS["twitter"])
    text = filter_text_chars(text, keep_tokens=True)
    text = normalize_token_spacing(text)
    return text

def preprocess_manual_youtube(text):
    text = safe_str(text)
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = strip_html_like(text)
    text = remove_invisible_and_control_chars(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = remove_hashtags(text)
    text = remove_emojis(text)
    text = remove_diacritics(text)
    text = normalize_alef(text)
    text = normalize_yeh(text)
    text = normalize_hamza(text)
    text = remove_tatweel(text)
    text = normalize_repetition(text, max_repeat=1)
    text = normalize_punctuation(text)
    text = collapse_visual_separators(text)
    text = handle_odd_chars(text, pattern=ODD_CHARS_PATTERNS["youtube"])
    text = remove_latin_spans(text)
    text = filter_text_chars(text, keep_tokens=False)
    text = normalize_token_spacing(text)
    return text


In [ ]:
def preprocess_deepfake_aware_youtube(text):
    text = safe_str(text)
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = strip_html_like(text)
    text = remove_invisible_and_control_chars(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = remove_hashtags(text)
    text = remove_emojis(text)
    text = replace_number_spans(text)
    text = remove_latin_spans(text)
    text = remove_diacritics(text)
    text = remove_tatweel(text)
    text = normalize_repetition_keep_tokens(text, max_repeat=2)
    text = normalize_punctuation(text)
    text = collapse_visual_separators(text)
    text = handle_odd_chars(text, pattern=ODD_CHARS_PATTERNS["youtube"])
    text = filter_text_chars(text, keep_tokens=True)
    text = normalize_token_spacing(text)
    return text

def clean_text_manual(text, platform):
    if platform == "twitter":
        return preprocess_manual_twitter(text)
    if platform == "youtube":
        return preprocess_manual_youtube(text)
    raise ValueError(f"Unsupported platform for manual preprocessing: {platform}")

def clean_text_deepfake_aware(text, platform):
    if platform == "twitter":
        return preprocess_deepfake_aware_twitter(text)
    if platform == "youtube":
        return preprocess_deepfake_aware_youtube(text)
    raise ValueError(f"Unsupported platform for deepfake-aware preprocessing: {platform}")

def apply_preprocessing(text, method, platform):
    if method == "original":
        return safe_str(text)
    if method == "manual":
        return clean_text_manual(text, platform=platform)
    if method == "deepfake_aware":
        return clean_text_deepfake_aware(text, platform=platform)
    return safe_str(text)

print("Preparing preprocessed text variants for both source platforms...")

preprocessed_datasets = {"twitter": {}, "youtube": {}}

for platform in ("twitter", "youtube"):
    preprocessed_datasets[platform]["original"] = {name: df["text"].tolist() for name, df in datasets.items()}
    preprocessed_datasets[platform]["manual"] = {
        name: [apply_preprocessing(text, "manual", platform=platform) for text in df["text"].tolist()]
        for name, df in datasets.items()
    }
    preprocessed_datasets[platform]["deepfake_aware"] = {
        name: [apply_preprocessing(text, "deepfake_aware", platform=platform) for text in df["text"].tolist()]
        for name, df in datasets.items()
    }

true_labels = {name: df["label"].astype(int).tolist() for name, df in datasets.items()}

print("Done preprocessing.")


In [ ]:
# 4. Configuration of Best Models
BEST_MODELS = [
    # Twitter Models
    {"platform": "twitter", "family": "Transformer", "preprocessing": "deepfake_aware", "model": "marbertv2", "embedding": "marbertv2"},
    {"platform": "twitter", "family": "Sequence", "preprocessing": "deepfake_aware", "model": "lstm", "embedding": "fasttext"},
    {"platform": "twitter", "family": "ML", "preprocessing": "deepfake_aware", "model": "linearsvc", "embedding": "tfidf"},

    {"platform": "twitter", "family": "Transformer", "preprocessing": "manual", "model": "marbertv2", "embedding": "marbertv2"},
    {"platform": "twitter", "family": "Sequence", "preprocessing": "manual", "model": "bilstm", "embedding": "fasttext"},
    {"platform": "twitter", "family": "ML", "preprocessing": "manual", "model": "linearsvc", "embedding": "tfidf"},

    {"platform": "twitter", "family": "Transformer", "preprocessing": "original", "model": "camelbert", "embedding": "camelbert"},
    {"platform": "twitter", "family": "Sequence", "preprocessing": "original", "model": "bilstm", "embedding": "fasttext"},
    {"platform": "twitter", "family": "ML", "preprocessing": "original", "model": "linearsvc", "embedding": "fasttext"},

    # YouTube Models
    {"platform": "youtube", "family": "Transformer", "preprocessing": "deepfake_aware", "model": "marbertv2", "embedding": "marbertv2"},
    {"platform": "youtube", "family": "Sequence", "preprocessing": "deepfake_aware", "model": "bilstm", "embedding": "fasttext"},
    {"platform": "youtube", "family": "ML", "preprocessing": "deepfake_aware", "model": "linearsvc", "embedding": "fasttext"},

    {"platform": "youtube", "family": "Transformer", "preprocessing": "manual", "model": "marbertv2", "embedding": "marbertv2"},
    {"platform": "youtube", "family": "Sequence", "preprocessing": "manual", "model": "bilstm", "embedding": "fasttext"},
    {"platform": "youtube", "family": "ML", "preprocessing": "manual", "model": "logisticregression", "embedding": "fasttext"},

    {"platform": "youtube", "family": "Transformer", "preprocessing": "original", "model": "arabertv2", "embedding": "arabertv2"},
    {"platform": "youtube", "family": "Sequence", "preprocessing": "original", "model": "bilstm", "embedding": "fasttext"},
    {"platform": "youtube", "family": "ML", "preprocessing": "original", "model": "logisticregression", "embedding": "fasttext"},
]


In [ ]:
# 5. Evaluation Engine
def evaluate_predictions(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }

all_results = []


In [ ]:
# 6. Checkpoint Resolution & Shared Helpers

def _safe_lower_path(path):
    return str(path).replace("\\", "/").lower()

def _is_hf_checkpoint_dir(path):
    return (
        path.is_dir()
        and (path / "config.json").exists()
        and ((path / "pytorch_model.bin").exists() or len(list(path.glob("*.safetensors"))) > 0)
    )

def _resolve_transformer_checkpoint_dir(config):
    platform = config["platform"]
    preprocessing = config["preprocessing"]
    model_name = config["model"]
    run_id = f"prep={preprocessing}__maxlen=128__seed=42"

    models_root = OUTPUT_DIR / platform / "models"
    direct_candidates = [
        models_root / model_name / run_id,
        models_root / model_name / run_id / "best",
        models_root / f"{model_name}_{platform}_{preprocessing}_{run_id}",
        models_root / f"{model_name}_{platform}_{preprocessing}",
        models_root / f"{model_name}_{platform}_{preprocessing}" / "best",
    ]
    for candidate in direct_candidates:
        if _is_hf_checkpoint_dir(candidate):
            return candidate

    discovered = []
    if models_root.exists():
        for cfg in models_root.rglob("config.json"):
            cdir = cfg.parent
            low = _safe_lower_path(cdir)
            if platform in low and preprocessing in low and model_name in low:
                if _is_hf_checkpoint_dir(cdir):
                    discovered.append(cdir)
                elif _is_hf_checkpoint_dir(cdir / "best"):
                    discovered.append(cdir / "best")
    if not discovered:
        raise FileNotFoundError(f"Could not find transformer checkpoint for {config}")

    discovered = sorted(
        set(discovered),
        key=lambda p: (0 if "/best" in _safe_lower_path(p) else 1, len(str(p)), str(p)),
    )
    return discovered[0]


In [ ]:
def _resolve_ml_joblib_path(config):
    platform = config["platform"]
    preprocessing = config["preprocessing"]
    model_name = config["model"]
    embedding = config["embedding"]
    ml_root = OUTPUT_DIR / platform / "ml_classifiers"
    direct = ml_root / "per_classifier" / f"{model_name}__{embedding}__{preprocessing}.joblib"
    if direct.exists():
        return direct

    expected_name = f"{model_name}__{embedding}__{preprocessing}.joblib".lower()
    candidates = []
    if ml_root.exists():
        for path in ml_root.rglob("*.joblib"):
            low_name = path.name.lower()
            low_path = _safe_lower_path(path)
            if low_name == expected_name:
                candidates.append(path)
            elif model_name in low_path and embedding in low_path and preprocessing in low_path:
                candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f"Could not find ML checkpoint for {config}")
    return sorted(set(candidates), key=lambda p: (0 if "per_classifier" in _safe_lower_path(p) else 1, len(str(p)), str(p)))[0]

def _resolve_embedding_model_path(platform, preprocessing, embedding_name):
    emb_root = OUTPUT_DIR / platform / "embeddings"
    direct_candidates = [
        emb_root / preprocessing / embedding_name / "model.bin",
        emb_root / embedding_name / preprocessing / "model.bin",
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate

    discovered = []
    if emb_root.exists():
        for path in emb_root.rglob("model.bin"):
            low = _safe_lower_path(path)
            if preprocessing in low and embedding_name in low:
                discovered.append(path)
    if not discovered:
        raise FileNotFoundError(
            f"Could not find embedding model.bin for platform={platform}, preprocessing={preprocessing}, embedding={embedding_name}"
        )
    return sorted(set(discovered), key=lambda p: (len(str(p)), str(p)))[0]

def tokenize_for_embedding(text):
    text = str(text or "").strip()
    return [tok for tok in text.split() if tok]

def get_vector_from_static_model(model, token, embedding_name):
    if embedding_name == "fasttext":
        return np.asarray(model.wv.get_vector(token), dtype=np.float32)
    if token in model.wv:
        return np.asarray(model.wv[token], dtype=np.float32)
    return None

def mean_pool_document(tokens, model, embedding_name):
    vectors = []
    for token in tokens:
        vec = get_vector_from_static_model(model, token, embedding_name)
        if vec is not None:
            vectors.append(vec)
    if not vectors:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(np.vstack(vectors), axis=0).astype(np.float32)

def vectorize_documents(tokenized_texts, model, embedding_name):
    return np.vstack([mean_pool_document(tokens, model, embedding_name) for tokens in tokenized_texts]).astype(np.float32)


In [ ]:
_embedding_model_cache = {}

def load_embedding_model(platform, preprocessing, embedding_name):
    cache_key = (platform, preprocessing, embedding_name)
    if cache_key in _embedding_model_cache:
        return _embedding_model_cache[cache_key]
    model_path = _resolve_embedding_model_path(platform, preprocessing, embedding_name)
    if embedding_name == "fasttext":
        model = FastText.load(str(model_path))
    elif embedding_name == "word2vec_cbow":
        model = Word2Vec.load(str(model_path))
    else:
        raise ValueError(f"Unsupported static embedding for ML evaluation: {embedding_name}")
    _embedding_model_cache[cache_key] = model
    return model

# Sequence helpers copied from training notebooks

SEQ_EMBED_DIM = 128

SEQ_HIDDEN_UNITS = 64

SEQ_DROPOUT = 0.25

SEQ_PAD_IDX = 0

SEQ_OOV_IDX = 1

class BaseSequenceClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_units, dropout, pad_idx, bidirectional=False, embedding_matrix=None, freeze_embeddings=False):
        super().__init__()
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.embedding.weight.requires_grad = not freeze_embeddings
        self.encoder = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_units,
            batch_first=True,
            bidirectional=bidirectional,
        )
        out_dim = hidden_units * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(out_dim, 2)

    def forward(self, tokens, lengths):
        embedded = self.embedding(tokens)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.encoder(packed)
        if self.bidirectional:
            hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            hidden = hidden[-1]
        return self.classifier(self.dropout(hidden))


In [ ]:
def instantiate_sequence_model(architecture, vocab_size, embedding_matrix):
    architecture = str(architecture).lower()
    if architecture == "lstm":
        return BaseSequenceClassifier(
            vocab_size=vocab_size,
            embed_dim=SEQ_EMBED_DIM,
            hidden_units=SEQ_HIDDEN_UNITS,
            dropout=SEQ_DROPOUT,
            pad_idx=SEQ_PAD_IDX,
            bidirectional=False,
            embedding_matrix=embedding_matrix,
            freeze_embeddings=False,
        )
    if architecture == "bilstm":
        return BaseSequenceClassifier(
            vocab_size=vocab_size,
            embed_dim=SEQ_EMBED_DIM,
            hidden_units=SEQ_HIDDEN_UNITS,
            dropout=SEQ_DROPOUT,
            pad_idx=SEQ_PAD_IDX,
            bidirectional=True,
            embedding_matrix=embedding_matrix,
            freeze_embeddings=False,
        )
    raise ValueError(f"Unsupported sequence architecture: {architecture}")

def load_sequence_checkpoint(path):
    payload = torch.load(path, map_location="cpu")
    if isinstance(payload, dict) and "state_dict" in payload:
        return payload["state_dict"], payload.get("metadata", {})
    if isinstance(payload, dict):
        return payload, {}
    raise ValueError(f"Unexpected sequence checkpoint format at {path}")

def encode_tokenized_texts(tokenized_texts, word_to_idx, max_len):
    encoded = np.full((len(tokenized_texts), max_len), SEQ_PAD_IDX, dtype=np.int64)
    lengths = np.zeros(len(tokenized_texts), dtype=np.int64)
    for i, tokens in enumerate(tokenized_texts):
        ids = [word_to_idx.get(token, SEQ_OOV_IDX) for token in tokens[:max_len]]
        lengths[i] = max(1, len(ids))
        if ids:
            encoded[i, :len(ids)] = ids
    return encoded, lengths

class SequenceInferenceDataset(TorchDataset):
    def __init__(self, X, lengths):
        self.X = torch.tensor(X, dtype=torch.long)
        self.lengths = torch.tensor(lengths, dtype=torch.long)

    def __len__(self):
        return len(self.lengths)

    def __getitem__(self, idx):
        return self.X[idx], self.lengths[idx]


In [ ]:
def _resolve_sequence_run_dir(config):
    platform = config["platform"]
    preprocessing = config["preprocessing"]
    architecture = config["model"]
    embedding = config["embedding"]
    seq_root = OUTPUT_DIR / platform / "sequence_models"

    direct_candidates = [
        seq_root / preprocessing / f"{architecture}__{embedding}",
        seq_root / preprocessing / architecture,
        seq_root / architecture / preprocessing,
        seq_root / preprocessing / f"{architecture}_{embedding}",
    ]
    for candidate in direct_candidates:
        if (candidate / "best_model.pt").exists():
            return candidate

    discovered = []
    if seq_root.exists():
        for ckpt in seq_root.rglob("best_model.pt"):
            low = _safe_lower_path(ckpt.parent)
            if preprocessing in low and architecture in low:
                discovered.append(ckpt.parent)
    if not discovered:
        raise FileNotFoundError(f"Could not find sequence checkpoint for {config}")

    embedding_hits = [d for d in discovered if embedding in _safe_lower_path(d)]
    candidates = embedding_hits if embedding_hits else discovered
    return sorted(set(candidates), key=lambda p: (len(str(p)), str(p)))[0]

def load_sequence_model_for_config(config):
    run_dir = _resolve_sequence_run_dir(config)
    checkpoint_path = run_dir / "best_model.pt"
    state_dict, checkpoint_meta = load_sequence_checkpoint(checkpoint_path)

    metadata = {}
    metadata_path = run_dir / "metadata.json"
    if metadata_path.exists():
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)
    if isinstance(checkpoint_meta, dict):
        metadata = {**metadata, **checkpoint_meta}

    vocab_path = run_dir / "vocab.json"
    if not vocab_path.exists():
        raise FileNotFoundError(f"Missing vocab.json next to sequence checkpoint: {vocab_path}")
    with open(vocab_path, "r", encoding="utf-8") as f:
        raw_vocab = json.load(f)
    word_to_idx = {str(k): int(v) for k, v in raw_vocab.items()}

    max_len = int(metadata.get("max_len", 128))
    vocab_size = int(metadata.get("vocab_size", len(word_to_idx)))

    embedding_matrix = None
    embedding_matrix_path = metadata.get("embedding_matrix_path")
    if embedding_matrix_path and Path(embedding_matrix_path).exists():
        embedding_matrix = np.load(embedding_matrix_path)
    elif (run_dir / "embedding_matrix.npy").exists():
        embedding_matrix = np.load(run_dir / "embedding_matrix.npy")

    if embedding_matrix is not None and embedding_matrix.shape[0] != vocab_size:
        vocab_size = int(embedding_matrix.shape[0])

    model = instantiate_sequence_model(config["model"], vocab_size, embedding_matrix)
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()
    return model, word_to_idx, max_len, run_dir


In [ ]:
def predict_sequence_labels(model, texts, word_to_idx, max_len=128, batch_size=128):
    tokenized = [tokenize_for_embedding(text) for text in texts]
    X, lengths = encode_tokenized_texts(tokenized, word_to_idx, max_len=max_len)
    loader = DataLoader(SequenceInferenceDataset(X, lengths), batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for tokens, lens in loader:
            tokens = tokens.to(DEVICE)
            lens = lens.to(DEVICE)
            logits = model(tokens, lens)
            batch_preds = torch.argmax(logits, dim=1).detach().cpu().numpy().tolist()
            preds.extend(batch_preds)
    return np.asarray(preds, dtype=int)


In [ ]:
# 7. Evaluate Transformers
for config in BEST_MODELS:
    if config["family"] != "Transformer":
        continue

    print(f"\nEvaluating Transformer: {config['platform']} | {config['preprocessing']} | {config['model']}")
    try:
        model_dir = _resolve_transformer_checkpoint_dir(config)
        print(f"  Checkpoint: {model_dir}")
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
        model.eval()

        for ds_name, texts in preprocessed_datasets[config["platform"]][config["preprocessing"]].items():
            y_true = true_labels[ds_name]
            y_pred = []
            batch_size = 32
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i : i + batch_size]
                inputs = tokenizer(
                    batch_texts,
                    padding=True,
                    truncation=True,
                    max_length=128,
                    return_tensors="pt",
                ).to(DEVICE)
                with torch.no_grad():
                    outputs = model(**inputs)
                    preds = torch.argmax(outputs.logits, dim=1).detach().cpu().numpy().tolist()
                y_pred.extend(preds)

            metrics = evaluate_predictions(y_true, y_pred)
            all_results.append({**config, "target_dataset": ds_name, **metrics})
            print(f"  [{ds_name}] F1: {metrics['f1']:.4f} | Acc: {metrics['accuracy']:.4f}")
    except Exception as e:
        print(f"  Failed: {e}")


In [ ]:
# 8. Evaluate ML Classifiers (TF-IDF + FastText/Word2Vec)
for config in BEST_MODELS:
    if config["family"] != "ML":
        continue

    print(f"\nEvaluating ML: {config['platform']} | {config['preprocessing']} | {config['model']} ({config['embedding']})")
    try:
        model_path = _resolve_ml_joblib_path(config)
        print(f"  Checkpoint: {model_path}")
        payload = joblib.load(model_path)
        clf = payload.get("estimator", payload) if isinstance(payload, dict) else payload

        for ds_name, texts in preprocessed_datasets[config["platform"]][config["preprocessing"]].items():
            y_true = true_labels[ds_name]
            embedding_name = config["embedding"]

            if embedding_name == "tfidf":
                if not isinstance(payload, dict) or "vectorizer" not in payload:
                    raise KeyError(f"TF-IDF vectorizer missing in joblib payload: {model_path}")
                X = payload["vectorizer"].transform(texts)
            else:
                emb_model = load_embedding_model(
                    platform=config["platform"],
                    preprocessing=config["preprocessing"],
                    embedding_name=embedding_name,
                )
                tokenized = [tokenize_for_embedding(text) for text in texts]
                X = vectorize_documents(tokenized, emb_model, embedding_name)

            y_pred = clf.predict(X)
            metrics = evaluate_predictions(y_true, y_pred)
            all_results.append({**config, "target_dataset": ds_name, **metrics})
            print(f"  [{ds_name}] F1: {metrics['f1']:.4f} | Acc: {metrics['accuracy']:.4f}")
    except Exception as e:
        print(f"  Failed: {e}")


In [ ]:
# 9. Evaluate Sequence Models (LSTM / BiLSTM)
for config in BEST_MODELS:
    if config["family"] != "Sequence":
        continue

    print(f"\nEvaluating Sequence: {config['platform']} | {config['preprocessing']} | {config['model']} ({config['embedding']})")
    try:
        model, word_to_idx, max_len, run_dir = load_sequence_model_for_config(config)
        print(f"  Checkpoint run dir: {run_dir}")
        print(f"  Vocab size: {len(word_to_idx)} | max_len: {max_len}")

        for ds_name, texts in preprocessed_datasets[config["platform"]][config["preprocessing"]].items():
            y_true = true_labels[ds_name]
            y_pred = predict_sequence_labels(
                model=model,
                texts=texts,
                word_to_idx=word_to_idx,
                max_len=max_len,
                batch_size=128,
            )
            metrics = evaluate_predictions(y_true, y_pred)
            all_results.append({**config, "target_dataset": ds_name, **metrics})
            print(f"  [{ds_name}] F1: {metrics['f1']:.4f} | Acc: {metrics['accuracy']:.4f}")
    except Exception as e:
        print(f"  Failed: {e}")


In [ ]:
# 10. Save and Preview Results
df_results = pd.DataFrame(all_results)
out_dir = OUTPUT_DIR / "output_for_paper" / "4_cross_domain_evaluation"
out_dir.mkdir(parents=True, exist_ok=True)

if not df_results.empty:
    csv_path = out_dir / "cross_domain_results_all_variants.csv"
    df_results.to_csv(csv_path, index=False)
    print(f"\nResults saved to: {csv_path}")
    display(
        df_results.sort_values(by=["target_dataset", "family", "f1"], ascending=[True, True, False]).head(30)
    )
else:
    print("\nNo results were generated. Check failed checkpoints above.")
